# Day 29 — Practice SOLUTIONS
### Correlation · Module 2 · Topic 2.9

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

Three questions to keep asking yourself:

- **Did I plot it?**
- **How many pairs did I look at?**
- **Which of the five explanations is this?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

print("ready")

---
# Part A — Predict the output

Eight short cells. Write your prediction down first, then run it. The point is to find out
where your intuition about r is wrong, so a wrong guess is more useful than a right one.

### A1 · Does rescaling y change r?

`y` is multiplied by 100 and shifted by 500. Predict all three numbers.

In [ ]:
rng = np.random.default_rng(101)
x = rng.normal(0, 1, 50)
y = 2*x + rng.normal(0, 1, 50)

print(round(stats.pearsonr(x, y).statistic, 4))
print(round(stats.pearsonr(x, 100*y + 500).statistic, 4))
print(round(stats.pearsonr(x, -y).statistic, 4))

**0.8745, 0.8745, −0.8745.**

r is computed after both variables are divided by their own standard deviation, so any
**positive** linear rescaling of either variable leaves it completely unchanged. Multiplying
by a negative number flips the sign and nothing else.

This is exactly why **r is not a slope**: the third dataset has a slope 100 times steeper
than the first and the same r.

### A2 · A perfect, noiseless parabola

There is no randomness here at all. `y` is determined by `x` exactly. What is r?

In [ ]:
x = np.linspace(-5, 5, 101)
print(round(stats.pearsonr(x, x**2).statistic, 4))

x2 = np.linspace(0, 5, 101)
print(round(stats.pearsonr(x2, x2**2).statistic, 4))

**0.0 and 0.9677.**

The first is the headline result of the whole session: a **perfect**, noiseless relationship
with r = 0. Over a symmetric range the left arm of the parabola goes down exactly as much as
the right arm goes up, and the products cancel.

The second line shows that r is not a property of the *relationship* — it is a property of
the relationship **and the range you sampled**. Take only the right-hand half and the same
parabola is monotonic, so r jumps to 0.97.

⚠️ Restricting the range of x is one of the quiet ways correlations change size in real work.

### A3 · Which way round?

In [ ]:
rng = np.random.default_rng(103)
a = rng.normal(0, 1, 200)
b = 0.7*a + rng.normal(0, 0.7, 200)

print(stats.pearsonr(a, b).statistic == stats.pearsonr(b, a).statistic)

**True** — exactly, bit for bit.

Correlation is **symmetric**. There is no sense in which `r(a, b)` is "the effect of a on b".
The direction of causation is simply not present in the number, which is why **reverse
causation** is one of the five explanations you have to rule out by argument, never by
computation.

### A4 · A steep but perfectly monotonic curve

In [ ]:
x = np.linspace(1, 5, 60)
y = x**5                       # no noise at all

print("Pearson :", round(stats.pearsonr(x, y).statistic, 4))
print("Spearman:", round(stats.spearmanr(x, y).statistic, 4))

**Pearson 0.8732, Spearman 1.0000.**

y is a perfect function of x — knowing x gives you y exactly — and Pearson reports 0.87,
which a reader would call "strong but imperfect". Spearman reports exactly 1, which is the
truth: the relationship is perfectly **monotonic**.

Pearson is not wrong. It is answering the question *"how close is this to a straight line?"*,
and a fifth-power curve is not close to a straight line. It is just not the question you
wanted answered.

### A5 · Three correlations, three shares of variance

In [ ]:
for r in [0.4, 0.6, 0.8]:
    print(f"r = {r}  ->  r^2 = {r**2:.0%}")

**16%, 36%, 64%.**

The relationship is quadratic, which is why doubling r from 0.4 to 0.8 **quadruples** the
variance explained, from 16% to 64%.

The practical consequence: r = 0.4 and r = 0.6 sound like neighbours and are not. 36% is more
than twice as much explanatory power as 16%.

### A6 · One point, added to a strong negative relationship

In [ ]:
rng = np.random.default_rng(66)
x = rng.normal(0, 1, 30)
y = -x + rng.normal(0, 0.4, 30)

print("30 points      :", round(stats.pearsonr(x, y).statistic, 4))
print("+ one at (10,10):", round(stats.pearsonr(np.append(x, 10), np.append(y, 10)).statistic, 4))

**−0.9348 becomes +0.5440.**

Not weakened — **reversed**. One point out of 31 turned a strong negative relationship into
a moderate positive one. Anyone reading only the second number would describe the data as
the exact opposite of what it is.

Pearson's r is built from squared distances from the mean, so a point far from the centre
of the cloud dominates the sum. With n = 31 a single value at (10, 10) outweighs the other
thirty combined.

### A7 · Two edge cases

In [ ]:
x = np.array([1., 2, 3, 4, 5])
c = np.array([7., 7, 7, 7, 7])

print("x with itself  :", stats.pearsonr(x, x).statistic)
print("x with constant:", stats.pearsonr(x, c).statistic)

**1.0 and `nan`.**

A constant has zero standard deviation, and r divides by it — so the answer is undefined,
not zero. scipy returns `nan` (with a warning).

⚠️ This matters because a column that is constant *in the rows you kept* is common: a filter,
a single-site subset, a status flag that never changes in this extract. `df.corr()` quietly
fills those cells with `NaN`, and a heatmap renders them as blanks that are easy to overlook.

### A8 · What is Spearman, really?

In [ ]:
rng = np.random.default_rng(8)
a = rng.normal(0, 1, 40)
b = np.exp(a)

print(round(stats.pearsonr(stats.rankdata(a), stats.rankdata(b)).statistic, 6))
print(round(stats.spearmanr(a, b).statistic, 6))

**1.0 and 1.0** — the same number, because they are the same calculation.

`spearmanr(a, b)` *is* `pearsonr(rankdata(a), rankdata(b))`. There is no separate theory to
learn: Spearman is Pearson applied to positions instead of values.

Both are exactly 1 here because `exp` is strictly increasing, so the ranks of `b` are
identical to the ranks of `a`.

---
# Part B — Spot & fix the bug

Each cell runs without error and produces a confident, wrong conclusion. Find it, say why,
and write the fix.

### B1 · "There is no relationship between dose and effect"

A pharmacologist runs this and concludes the drug does nothing.

In [ ]:
rng = np.random.default_rng(310)
dose   = rng.uniform(0, 10, 120)
effect = -(dose - 5)**2 + 25 + rng.normal(0, 2.5, 120)

res = stats.pearsonr(dose, effect)
print(f"r = {res.statistic:.4f}, p = {res.pvalue:.4f}")
print("Conclusion: no relationship between dose and effect.")

**What is wrong, and what is the real finding?**

In [ ]:
# THE FIX: plot it. Four seconds.
plt.figure(figsize=(6, 3))
plt.scatter(dose, effect, s=14, alpha=.8)
plt.xlabel("dose"); plt.ylabel("effect")
plt.tight_layout(); plt.show()

print("Pearson :", round(stats.pearsonr(dose, effect).statistic, 4))
print("Spearman:", round(stats.spearmanr(dose, effect).statistic, 4), " <- no help either")

low, high = dose < 5, dose >= 5
print("\nwithin the low half  : r = %+.3f" % stats.pearsonr(dose[low],  effect[low]).statistic)
print("within the high half : r = %+.3f" % stats.pearsonr(dose[high], effect[high]).statistic)

**The bug: nobody plotted it.** The relationship is a near-perfect inverted U — a textbook
dose-response curve with an optimum at dose = 5. Pearson reports r = −0.123, p = 0.18, and
the pharmacologist shelves a working drug.

Note that **Spearman does not rescue this** (−0.140). A U-shape goes up and then down, so
the ranks do not line up either. There is no coefficient that fixes this; only the plot does.

**The real finding, once you split at the peak:**

- within the low half, r = **+0.923** — more dose, more effect
- within the high half, r = **−0.894** — more dose, less effect

The honest report is *"effect peaks at a dose of about 5 and falls away on both sides"*, and
the right analysis is a curve fit, not a correlation.

### B2 · "We found a significant driver of churn"

An analyst correlates every pair of twelve metrics and reports the strongest.

In [ ]:
rng = np.random.default_rng(311)
df = pd.DataFrame(rng.normal(0, 1, (50, 12)),
                  columns=[f"m{i:02d}" for i in range(12)])
# (all twelve columns are independent noise)

best = None
for i in range(12):
    for j in range(i + 1, 12):
        res = stats.pearsonr(df.iloc[:, i], df.iloc[:, j])
        if best is None or res.pvalue < best[3]:
            best = (df.columns[i], df.columns[j], res.statistic, res.pvalue)

print(f"{best[0]} and {best[1]}: r = {best[2]:.3f}, p = {best[3]:.5f}")
print("Significant at p < 0.05 -> worth investigating.")

**What is wrong, and what is the correct threshold?**

In [ ]:
n_pairs = 12 * 11 // 2
print("pairs silently tested :", n_pairs)
print("expected 'significant' by chance alone : %.1f" % (n_pairs * 0.05))
print("Bonferroni threshold  : %.5f" % (0.05 / n_pairs))
print("the 'finding' p       : %.5f" % best[3])
print("survives correction?  :", best[3] < 0.05 / n_pairs)

**The bug: the loop ran 66 hypothesis tests and reported only the winner.** The p-value
printed (0.00739) is the p-value for *one* pre-specified pair, not for "the largest of 66".

With 66 tests you expect about **3.3** to clear p < 0.05 even when, as here, every column is
independent noise.

**The fix:** the Bonferroni threshold is 0.05 / 66 = **0.00076**, and p = 0.00739 does not
come close. The correct conclusion is "nothing here", which is also the truth.

A correlation matrix is a tool for **generating** hypotheses. If you found the pair by
looking, you need either a correction or — better — a fresh dataset to test it on.

### B3 · "Strong evidence from our pilot study"

Eight customers were surveyed. The consultant reports the result.

In [ ]:
rng = np.random.default_rng(537)
satisfaction = rng.normal(0, 1, 8)
spend        = rng.normal(0, 1, 8)     # genuinely unrelated

res = stats.pearsonr(satisfaction, spend)
print(f"r = {res.statistic:.3f}, p = {res.pvalue:.4f}")
print("Satisfaction strongly predicts spend (p < 0.05).")

**What is missing, and what should the report say?**

In [ ]:
res = stats.pearsonr(satisfaction, spend)
lo, hi = res.confidence_interval()
print("n        = 8")
print("r        = %+.3f" % res.statistic)
print("95%% CI   = [%+.3f, %+.3f]   <- width %.2f" % (lo, hi, hi - lo))
print("r^2      = %.0f%%" % (res.statistic**2 * 100))
print("\nThe interval spans almost the whole possible range of r.")

**The bug: no confidence interval, and n = 8.**

r = 0.713 with p = 0.047 looks like a finding. The 95% interval is **[+0.017, +0.944]** —
it very nearly touches zero at one end and almost perfect at the other. The data is
consistent with "essentially no relationship" and with "almost deterministic".

The data was in fact generated with **no relationship at all**. This is a 1-in-20 event, and
with a pilot study of eight people you will see one before long.

**What the report should say:** *"In a pilot of 8 customers, satisfaction and spend
correlated at r = 0.71 (95% CI 0.02 to 0.94). The interval is far too wide to act on; a
sample of at least 50 would be needed to estimate this to useful precision."*

### B4 · "Coffee machines drive revenue"

In [ ]:
rng = np.random.default_rng(313)
n = 300
company_size = rng.uniform(1, 10, n)                       # in hundreds of staff
coffee_machines = 3*company_size + rng.normal(0, 2, n)
revenue         = 50*company_size + rng.normal(0, 30, n)

res = stats.pearsonr(coffee_machines, revenue)
print(f"r = {res.statistic:.3f}, p = {res.pvalue:.2e}")
print("Recommendation: install more coffee machines to increase revenue.")

**Which of the five explanations is this, and how do you demonstrate it?**

In [ ]:
def partial_corr(a, b, control):
    ra = a - np.polyval(np.polyfit(control, a, 1), control)
    rb = b - np.polyval(np.polyfit(control, b, 1), control)
    return stats.pearsonr(ra, rb)

raw = stats.pearsonr(coffee_machines, revenue)
pc  = partial_corr(coffee_machines, revenue, company_size)

print("raw r                              : %+.4f  (p = %.2e)" % (raw.statistic, raw.pvalue))
print("partial r, controlling company size: %+.4f  (p = %.3f)" % (pc.statistic, pc.pvalue))

**Explanation 2: confounding.** Company size causes both. Big companies have more coffee
machines *and* more revenue; the machines do nothing.

r = 0.948 with p ≈ 1e-150 — about as strong as correlations ever get — and the causal claim
is still nonsense. **Strength of correlation is not evidence of causation, and neither is a
tiny p-value.**

Controlling for company size drops the partial correlation to **+0.009 (p = 0.88)**. Once you
know how big the company is, the coffee machines tell you nothing more about revenue.

⚠️ The only reason this was demonstrable is that `company_size` happened to be in the
dataset. There is no procedure that controls for a confounder you did not measure — which is
the entire argument for randomised experiments.

---
# Part C — Write the code

Five tasks. Write the function, then run the test cell underneath it.

### C1 · A correlation report you could paste into a document

Write `report(x, y, n_pairs=1)` that prints **all five numbers** a reader needs: n, r with
its 95% confidence interval, Spearman, r², and the slope. It should also warn when Pearson
and Spearman disagree by more than 0.15.

In [ ]:
def report(x, y, n_pairs=1):
    # your code here
    pass

In [ ]:
def report(x, y, n_pairs=1, labels=("x", "y")):
    x, y = np.asarray(x, float), np.asarray(y, float)

    plt.figure(figsize=(5, 2.6))
    plt.scatter(x, y, s=16, alpha=.75)
    plt.xlabel(labels[0]); plt.ylabel(labels[1])
    plt.tight_layout(); plt.show()

    pe = stats.pearsonr(x, y)
    sp = stats.spearmanr(x, y)
    lo, hi = pe.confidence_interval()
    slope = np.polyfit(x, y, 1)[0]

    print(f"n        = {len(x)}")
    print(f"Pearson  = {pe.statistic:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")
    print(f"Spearman = {sp.statistic:+.3f}")
    print(f"r^2      = {pe.statistic**2:.1%} of the variance in {labels[1]}")
    print(f"slope    = {slope:+.3f} {labels[1]} per {labels[0]}")
    print(f"p        = {pe.pvalue:.2e}   threshold {0.05/n_pairs:.5f} ({n_pairs} pair(s))")
    if abs(pe.statistic - sp.statistic) > 0.15:
        print("  !! Pearson and Spearman disagree - check for outliers or a curve")

In [ ]:
rng = np.random.default_rng(340)
hrs = rng.gamma(4, 2.0, 70)
sc  = np.clip(45 + 3.0*hrs + rng.normal(0, 9, 70), 0, 100)
report(hrs, sc, n_pairs=1, labels=("hour", "mark"))

The plot comes **first**, before any number is printed. That ordering is the point of the
exercise: it makes the Anscombe check unavoidable rather than optional.

The disagreement check is the cheapest outlier detector you will ever write. Pearson and
Spearman answer almost the same question on clean, roughly linear data, so a large gap
between them means one of two things: a heavy outlier, or a curve.

### C2 · How much does any single point matter?

Write `most_influential(x, y)` that removes each point in turn, recomputes r, and reports
which single point moves r the most — and by how much.

In [ ]:
rng = np.random.default_rng(320)
x = np.append(rng.normal(0, 1, 60), 6.0)
y = np.append(0.1*rng.normal(0, 1, 60) + rng.normal(0, 1, 60), 6.0)
# note: x and y are essentially unrelated, except for that last point

def most_influential(x, y):
    # your code here
    pass

In [ ]:
def most_influential(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    full = stats.pearsonr(x, y).statistic
    deltas = np.array([full - stats.pearsonr(np.delete(x, i), np.delete(y, i)).statistic
                       for i in range(len(x))])
    k = int(np.argmax(np.abs(deltas)))
    return k, deltas[k], full, full - deltas[k]

rng = np.random.default_rng(320)
x = rng.normal(0, 1, 60)
y = 0.1*x + rng.normal(0, 1, 60)
x = np.append(x, 6.0); y = np.append(y, 6.0)

k, delta, full, without = most_influential(x, y)
print("r with everything       : %+.4f" % full)
print("most influential point  : index %d at (%.2f, %.2f)" % (k, x[k], y[k]))
print("r without it            : %+.4f" % without)
print("one point is worth      : %.4f of r" % abs(delta))

**r = +0.4201 with the point, +0.0860 without it.** One observation out of 61 is carrying
**0.334** of the correlation — nearly 80% of it.

A useful rule: if removing one point changes r by more than about 0.1, you do not have a
relationship, you have a point. Report the analysis both ways and say which observation it
was.

This is also the honest alternative to deleting outliers. Deleting a point because it hurts
your result is p-hacking; **reporting the result with and without it** is not.

### C3 · Partial correlation

Shoe size and reading ability in schoolchildren. Write `partial_corr(a, b, control)` and
use it to work out what is really going on.

In [ ]:
rng = np.random.default_rng(321)
n = 400
age     = rng.uniform(5, 16, n)
shoe    = 18 + 0.9*age + rng.normal(0, 0.8, n)
reading = 10 + 4.5*age + rng.normal(0, 6, n)

print("shoe vs reading: r = %.4f" % stats.pearsonr(shoe, reading).statistic)

def partial_corr(a, b, control):
    # your code here
    pass

In [ ]:
def partial_corr(a, b, control):
    """Correlation between a and b after removing the linear effect of `control`."""
    a, b, control = map(lambda v: np.asarray(v, float), (a, b, control))
    resid_a = a - np.polyval(np.polyfit(control, a, 1), control)
    resid_b = b - np.polyval(np.polyfit(control, b, 1), control)
    return stats.pearsonr(resid_a, resid_b)

raw = stats.pearsonr(shoe, reading)
pc  = partial_corr(shoe, reading, age)

print("raw                      : r = %+.4f  p = %.2e" % (raw.statistic, raw.pvalue))
print("controlling for age      : r = %+.4f  p = %.3f" % (pc.statistic, pc.pvalue))
print("\nr^2 fell from %.0f%% to %.1f%%" % (raw.statistic**2*100, pc.statistic**2*100))

**r = 0.9045 raw, +0.0833 (p = 0.096) once age is held still** — and r² falls from 82%
to 0.7%.

Bigger feet do not help anybody read. Older children have bigger feet and read better; age
causes both. This is the same shape as the ice cream and drownings example, and it is the
standard illustration of confounding for a reason — the raw correlation is very strong and
the mechanism is obviously absurd, which makes it easy to see. In real work the absurd
mechanism is usually plausible enough that nobody checks.

Note the partial correlation is not exactly zero (p = 0.096, which some people would be
tempted to call "marginally significant"). That is ordinary residual noise at n = 400 — and
with r² = 0.7%, even if it were real it would be worth nothing. **This is why r² goes next
to p.**

**How the function works:** fit a line predicting each variable from the confounder, throw
away the part that line explains, and correlate what is left. If the leftovers still move
together, there is something beyond the confounder. Here there is nothing.

### C4 · Scan a table for Pearson/Spearman disagreements

Write `disagreements(df, gap=0.15)` that compares the two coefficients for every pair of
columns and returns only the pairs where they differ by more than `gap`. This is a
one-glance data-quality check you can run on any new dataset.

In [ ]:
rng = np.random.default_rng(322)
df = pd.DataFrame({"a": rng.normal(0, 1, 80),
                   "b": rng.normal(0, 1, 80),
                   "c": rng.normal(0, 1, 80)})
df["d"] = np.exp(2*df.a)          # a monotonic curve
df["e"] = df.b.copy()
df.loc[0, "e"] = 25               # one bad value

def disagreements(df, gap=0.15):
    # your code here
    pass

In [ ]:
def disagreements(df, gap=0.15):
    cols, rows = list(df.columns), []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pe = stats.pearsonr(df[cols[i]], df[cols[j]]).statistic
            sp = stats.spearmanr(df[cols[i]], df[cols[j]]).statistic
            if abs(pe - sp) > gap:
                rows.append({"a": cols[i], "b": cols[j],
                             "pearson": round(pe, 3), "spearman": round(sp, 3),
                             "gap": round(abs(pe - sp), 3)})
    return pd.DataFrame(rows).sort_values("gap", ascending=False)

print(disagreements(df).to_string(index=False))
print("\npairs checked:", len(df.columns)*(len(df.columns)-1)//2)

Two pairs are flagged, and they are the two problems that were planted:

| pair | Pearson | Spearman | gap | what it is |
|---|---|---|---|---|
| **b–e** | +0.273 | +0.959 | 0.686 | one bad value (25) in a copy of `b` |
| **a–d** | +0.648 | +1.000 | 0.352 | `d = exp(2a)` — a perfect monotonic curve |

Both would be invisible in a plain correlation heatmap: b–e shows as a weak 0.27 and a–d as
a moderate 0.65, and neither looks remarkable.

Note the two different causes produce the same symptom. The gap tells you *something is off*,
not *which thing* — you still have to plot the flagged pair to see whether it is an outlier
or a curve.

### C5 · How much data do you actually need?

Two questions a project plan should answer before the data is collected:

1. What is the smallest n at which a true correlation of r would reach p < 0.05?
2. How many rows do you need for the 95% interval around r = 0.4 to be narrower than 0.20?

In [ ]:
def n_for_significance(r, alpha=0.05):
    # your code here
    pass

def ci_width(r, n):
    # your code here
    pass

In [ ]:
def n_for_significance(r, alpha=0.05):
    """Smallest n at which an observed |r| this size clears p < alpha."""
    for n in range(4, 500_000):
        t = stats.t.ppf(1 - alpha/2, n - 2)
        crit = np.sqrt(t**2 / (t**2 + n - 2))
        if crit <= abs(r):
            return n

def ci_width(r, n, conf=0.95):
    """Width of the Fisher-z confidence interval for r at sample size n."""
    z, se = np.arctanh(r), 1/np.sqrt(n - 3)
    c = stats.norm.ppf(1 - (1 - conf)/2)
    return np.tanh(z + c*se) - np.tanh(z - c*se)

print("to reach p < 0.05:")
for r in [0.1, 0.2, 0.3, 0.5]:
    print(f"   r = {r}  needs n = {n_for_significance(r)}")

print("\nwidth of the 95% CI around r = 0.4:")
for n in [30, 100, 300, 1000]:
    print(f"   n = {n:>4}  ->  width {ci_width(0.4, n):.3f}")

n = next(n for n in range(10, 100_000) if ci_width(0.4, n) <= 0.20)
print(f"\nn needed for a width of 0.20 at r = 0.4: {n}")

**To reach p < 0.05:** r = 0.1 needs **n = 385**; r = 0.2 needs **97**; r = 0.3 needs **44**;
r = 0.5 needs only **16**.

**Width of the 95% interval around r = 0.4:**

| n | width |
|---|---|
| 30 | 0.618 |
| 100 | 0.332 |
| 300 | 0.191 |
| 1000 | 0.104 |

You need **n = 273** for a width of 0.20.

The two questions have very different answers, and the gap between them is the point. At
n = 44 a correlation of 0.3 is "significant" — and its interval is roughly 0.01 to 0.55, which
is almost no information. **Significance is a much lower bar than precision.** Plan for the
interval you want, not for the p-value.

---
# Part D — Challenge

Two open tasks. Both are short to code and long to think about.

### D1 · Manufacture a finding, then take it apart

You have been handed 20 business KPIs measured over 60 weeks. There is **nothing** in the
data — every column is independent noise.

1. Find the most "significant" pair.
2. Write the two-sentence summary a careless analyst would put in a deck.
3. Write the honest version.
4. State exactly what extra information turns version 2 into version 3.

In [ ]:
rng = np.random.default_rng(330)
cols = [f"kpi_{i:02d}" for i in range(20)]
df = pd.DataFrame(rng.normal(0, 1, (60, 20)), columns=cols)

# your code here

In [ ]:
results = []
for i in range(20):
    for j in range(i + 1, 20):
        res = stats.pearsonr(df.iloc[:, i], df.iloc[:, j])
        results.append((cols[i], cols[j], res.statistic, res.pvalue))

out = pd.DataFrame(results, columns=["a", "b", "r", "p"]).sort_values("p")
n_pairs = len(out)

print("pairs tested          :", n_pairs)
print("p < 0.05              :", (out.p < 0.05).sum(), f"(expected {n_pairs*0.05:.1f} by chance)")
print("\nthe 'best' finding:")
print(out.head(3).to_string(index=False))

bonf = 0.05 / n_pairs
print("\nBonferroni threshold  : %.6f" % bonf)
print("survivors             :", (out.p < bonf).sum())

**The manufactured finding:** `kpi_04` and `kpi_13`, r = 0.409, p = 0.0012. Seventeen of the
190 pairs cleared p < 0.05 (chance alone predicts about 9.5 — the extra eight are ordinary
sampling variation, which is itself worth noticing).

**Version 2 — the careless summary:**

> *"We found a significant positive relationship between kpi_04 and kpi_13 (r = 0.41,
> p = 0.001). Teams that improve kpi_04 should expect gains in kpi_13."*

Every number in it is correct. It is still false.

**Version 3 — the honest version:**

> *"We screened all 190 pairwise correlations among 20 KPIs over 60 weeks. Seventeen reached
> p < 0.05, against roughly 10 expected by chance. The strongest (kpi_04 / kpi_13, r = 0.41,
> p = 0.0012) does not survive correction for 190 tests (threshold 0.00026), and r² = 17%
> with a 95% interval of roughly 0.17 to 0.60. We treat this as a hypothesis to test on
> next quarter's data, not as a finding."*

**What turns 2 into 3:** the **number of pairs examined**. That single fact — 190, not 1 —
changes the threshold by a factor of 190 and converts a discovery into noise. It is also the
one piece of information that is never visible in the output, which is why it has to be
reported deliberately. Nothing else about the analysis was wrong.

### D2 · Diagnose the explanation

A recruiting team reports: *"Among our hires, interview preparation is strongly **negatively**
correlated with years of experience (r = −0.65, p < 1e-42). Experienced candidates don't
bother preparing."*

The data below is the full applicant pool, with a `hired` flag.

1. Compute the correlation among the hired, and among everyone.
2. Decide which of the five explanations this is.
3. Say what the recruiting team should conclude instead.

In [ ]:
rng = np.random.default_rng(331)
N = 4000
prep       = rng.normal(0, 1, N)                          # interview preparation
experience = rng.normal(0, 1, N)                          # years of experience
hired      = (prep + experience + rng.normal(0, 0.3, N)) > 1.9

applicants = pd.DataFrame({"prep": prep, "experience": experience, "hired": hired})
print(applicants.hired.value_counts().to_dict())

# your code here

In [ ]:
h = applicants[applicants.hired]

r_all   = stats.pearsonr(applicants.prep, applicants.experience)
r_hired = stats.pearsonr(h.prep, h.experience)

print("EVERYONE   n = %4d   r = %+.4f   p = %.3f" % (len(applicants), r_all.statistic, r_all.pvalue))
print("HIRED ONLY n = %4d   r = %+.4f   p = %.2e" % (len(h), r_hired.statistic, r_hired.pvalue))
print("Spearman among the hired: %+.4f  (so it is not an outlier)"
      % stats.spearmanr(h.prep, h.experience).statistic)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].scatter(applicants.prep[~applicants.hired], applicants.experience[~applicants.hired],
                s=5, color="lightsteelblue")
axes[0].scatter(h.prep, h.experience, s=7, color="orangered")
axes[0].set_title(f"all applicants   r = {r_all.statistic:+.3f}", fontsize=11)
axes[1].scatter(h.prep, h.experience, s=12, color="orangered")
axes[1].set_title(f"hired only   r = {r_hired.statistic:+.3f}", fontsize=11)
for ax in axes:
    ax.set_xlabel("preparation"); ax.set_ylabel("experience")
plt.tight_layout(); plt.show()

**The numbers:** among all 4,000 applicants r = **−0.006** (p = 0.70) — no relationship
whatsoever. Among the 341 who were hired, r = **−0.653** (p = 9e-43). Spearman agrees at
−0.653, so this is not an outlier artefact. The correlation is entirely real *within the
sample the team looked at*.

**The explanation: 4 — selection** (specifically collider bias).

Being hired is a **common effect** of preparation and experience: you get in with enough of
either. Conditioning on a common effect induces a correlation between its causes. Once
someone is in the hired group, learning that they prepared very little tells you they must
have had the experience to compensate — otherwise they would not have been hired.

Nothing caused anything. The selection rule wrote the correlation into the data.

**What the team should conclude:** nothing about candidates. They are looking at their own
hiring bar reflected back at them. The claim *"experienced candidates don't bother preparing"*
is a statement about the applicant pool, and in the applicant pool the correlation is zero.

**The general lesson:** any analysis restricted to people who were **admitted, hired,
diagnosed, subscribed or survived** is at risk of this, and the risk is largest exactly when
the selection depends on both variables you are correlating. The only fix is to get the data
on the people who were *not* selected — which is why the question *"who is in this dataset,
and what rule put them here?"* is on the checklist.

---
### Before you hand in

- For every correlation you computed, did you **plot it first**?
- Did you report **r², the confidence interval and n**, not just r?
- Did you check **Spearman** alongside Pearson, and explain any gap?
- Did you count **how many pairs** you examined, and say so?
- For every causal-sounding claim, did you name which of the **five explanations** it is?
- For Part D, did you write both the misleading version and the honest one? Both are marked.

Solutions will be released after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*